In [ ]:
import numpy as np
import pandas as pd
import os

from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time
from tqdm import tqdm
import warnings
from scipy.integrate import IntegrationWarning
warnings.filterwarnings("ignore", category=IntegrationWarning)

In [7]:
def loo_rmst(df_group, rmst_group, follow_up):
    rmst_list = []
    group_size = len(df_group)
    for i in range(group_size):
        idx_to_drop = df_group.index[i]
        df_loo = df_group.drop(index=idx_to_drop)
        kmf = KaplanMeierFitter().fit(df_loo['time'], event_observed=df_loo['event'])
        rmst = group_size * rmst_group - (group_size-1) * restricted_mean_survival_time(kmf, t=follow_up, return_variance=False)
        rmst_list.append(rmst)
    return rmst_list

In [193]:
def loo_rmst_counter(df_group, rmst_group, follow_up):
    rmst_list = []
    group_size = len(df_group)
    for i in range(group_size):
        idx_to_drop = df_group.index[i]
        df_loo = df_group.drop(index=idx_to_drop)
        kmf = KaplanMeierFitter().fit(df_loo['counter_time'], event_observed=df_loo['counter_event'])
        rmst = group_size * rmst_group - (group_size-1) * restricted_mean_survival_time(kmf, t=follow_up, return_variance=False)
        rmst_list.append(rmst)
    return rmst_list

## Simulate df with varying treatment effect

In [ ]:
n_values =[30, 100, 1000]
fc = 0.4
follow_up = 1.5

for ve in [5,10,20,30,40,50,60,70,80,90]:
        df_sim = []
        estimations = []
        for n in n_values:
                folder_path = f"./sim_ct_df/sim_n{n}_fc{fc}_ve{ve}"
                files = sorted(os.listdir(folder_path))
                for file in tqdm(files, desc=f"Simulations for n={n} and vaccine efficacy={ve}%"):
                        df = pd.read_csv(os.path.join(folder_path, file))
                        df["n"] = n
                        df["sim"] = file.split('_')[1].split('.')[0]

                        # Compute outcome: leave-one-out RMST for each patient
                        kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time'], event_observed=df[df['group']==1]['event'])
                        kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time'], event_observed=df[df['group']==0]['event'])
                        kmf_trt_counter = KaplanMeierFitter().fit(df[df['group']==1]['counter_time'], event_observed=df[df['group']==1]['counter_event'])
                        rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                        rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)
                        rmst_active_counter = restricted_mean_survival_time(kmf_trt_counter, t=1.5, return_variance=False)

                        df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up)
                        df.loc[df['group'] == 1, 'LOO_RMST_SEP_COUNTER'] = loo_rmst_counter(df[df['group'] == 1], rmst_active_counter, follow_up)
                        df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up)
                        var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                        var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                        # Compute classic estimator
                        ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                        var_ate_rmst_manual = var_active/len(df[df['group']==1]) + var_placebo/len(df[df['group']==0])
                
                        for i in range(14):
                                noise = round(i * 0.05, 2)
                                df_tmp = df.copy()

                                # Preds Trt = counterfactual random by order    
                                df_treated = df_tmp[df_tmp['group'] == 1].copy()
                                df_placebo = df_tmp[df_tmp['group'] == 0].copy()    
                                counter_sorted = np.sort(df_treated['LOO_RMST_SEP_COUNTER'].values)
                                df_treated_sorted_by_sep = df_treated.sort_values('LOO_RMST_SEP')
                                df_treated_sorted_by_sep['PREDS_LOO_RMST_SHIFTED_ORDER'] = counter_sorted
                                df_treated_final = df_treated_sorted_by_sep.sort_index()
                                df_placebo['PREDS_LOO_RMST_SHIFTED_ORDER'] = df_placebo['LOO_RMST_SEP']
                                df_tmp = pd.concat([df_treated_final, df_placebo]).sort_index()

                                # Add swap-based noise -> random swap within group
                                preds = df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'].values.copy()
                                swap_fraction = noise
                                for g in [0, 1]: 
                                        idx = np.where(df_tmp['group'].values == g)[0]
                                        n = len(idx)

                                        num_swaps = int(n * swap_fraction)
                                        if num_swaps < 1:
                                                continue
                                        i1 = np.random.choice(idx, num_swaps, replace=False)
                                        i2 = np.random.choice(idx, num_swaps, replace=False)
                                        for a, b in zip(i1, i2):
                                                preds[a], preds[b] = preds[b], preds[a]

                                df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'] = preds
                                df_tmp['noise_level'] = noise
                                
                                # Compute PPCT estimator
                                sigma_f_2 = df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'].var()
                                sigma_t_2 = var_active
                                sigma_c_2 = var_placebo
                                rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_SHIFTED_ORDER'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                                rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_SHIFTED_ORDER'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                                lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                                var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                                        ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                                ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_SHIFTED_ORDER']).mean() - \
                                        (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_SHIFTED_ORDER']).mean()
                                r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'])**2
                                var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                                estimations.append({
                                        "n": n,
                                        "sim": file.split('_')[1].split('.')[0],
                                        "noise_level": noise,
                                        "ate_classic RMST": ate_rmst_manual,
                                        "var_classic RMST": var_ate_rmst_manual,
                                        "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                                        "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                                        "r2_shifted_order RMST": r_2_shifted_order,
                                        "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                                        "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                                        })
                                df_sim.append(df_tmp)

        df_sim = pd.concat(df_sim, ignore_index=True)
        estimations = pd.DataFrame(estimations)
        df_sim.to_csv(f"sim_outputs_1000/df_sim_fc{fc}_ve{ve}_counter.csv", index=False)
        estimations.to_csv(f"sim_outputs_1000/estimations_fc{fc}_ve{ve}_counter.csv", index=False)


Simulations for n=100 and vaccine efficacy=70%: 100%|██████████| 1000/1000 [14:08<00:00,  1.18it/s]
Simulations for n=1000 and vaccine efficacy=70%:   4%|▎         | 35/1000 [05:13<2:19:00,  8.64s/it]

## Simulate df with varying event probability in the control group

In [ ]:
n_values =[30, 100, 1000]
ve = 20
follow_up = 1.5

for fc in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        df_sim = []
        estimations = []
        for n in n_values:
                folder_path = f"./sim_ct_df/sim_n{n}_fc{fc}_ve{ve}"
                files = sorted(os.listdir(folder_path))
                for file in tqdm(files, desc=f"Simulations for n={n} , fc = {fc} , vaccine efficacy={ve}%"):
                        df = pd.read_csv(os.path.join(folder_path, file))
                        df["n"] = n
                        df["sim"] = file.split('_')[1].split('.')[0] 

                        # Compute outcome: leave-one-out RMST for each patient
                        kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time'], event_observed=df[df['group']==1]['event'])
                        kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time'], event_observed=df[df['group']==0]['event'])
                        kmf_trt_counter = KaplanMeierFitter().fit(df[df['group']==1]['counter_time'], event_observed=df[df['group']==1]['counter_event'])
                        rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                        rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)
                        rmst_active_counter = restricted_mean_survival_time(kmf_trt_counter, t=1.5, return_variance=False)

                        df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up)
                        df.loc[df['group'] == 1, 'LOO_RMST_SEP_COUNTER'] = loo_rmst_counter(df[df['group'] == 1], rmst_active_counter, follow_up)
                        df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up)
                        var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                        var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                        # Compute classic estimator
                        ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                        var_ate_rmst_manual = var_active/len(df[df['group']==1]) + var_placebo/len(df[df['group']==0])
                
                        for i in range(14):
                                noise = round(i * 0.05, 2)
                                df_tmp = df.copy()                           

                                # Preds Trt = counterfactual random from R by order
                                df_treated = df_tmp[df_tmp['group'] == 1].copy()
                                df_placebo = df_tmp[df_tmp['group'] == 0].copy()    
                                counter_sorted = np.sort(df_treated['LOO_RMST_SEP_COUNTER'].values)
                                df_treated_sorted_by_sep = df_treated.sort_values('LOO_RMST_SEP')
                                df_treated_sorted_by_sep['PREDS_LOO_RMST_SHIFTED_ORDER'] = counter_sorted
                                df_treated_final = df_treated_sorted_by_sep.sort_index()
                                df_placebo['PREDS_LOO_RMST_SHIFTED_ORDER'] = df_placebo['LOO_RMST_SEP']
                                df_tmp = pd.concat([df_treated_final, df_placebo]).sort_index()
                                
                                # Add swap-based noise -> random swap within group
                                preds = df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'].values.copy()
                                swap_fraction = noise
                                for g in [0, 1]: 
                                        idx = np.where(df_tmp['group'].values == g)[0]
                                        n = len(idx)

                                        num_swaps = int(n * swap_fraction)
                                        if num_swaps < 1:
                                                continue
                                        i1 = np.random.choice(idx, num_swaps, replace=False)
                                        i2 = np.random.choice(idx, num_swaps, replace=False)
                                        for a, b in zip(i1, i2):
                                                preds[a], preds[b] = preds[b], preds[a]

                                df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'] = preds
                                df_tmp['noise_level'] = noise
                                
                                # Compute PPCT estimator
                                sigma_f_2 = df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'].var()
                                sigma_t_2 = var_active
                                sigma_c_2 = var_placebo
                                rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_SHIFTED_ORDER'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                                rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_SHIFTED_ORDER'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                                lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                                #lambda_star_shifted_order = lambda_star
                                var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                                        ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                                ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_SHIFTED_ORDER']).mean() - \
                                        (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_SHIFTED_ORDER']).mean()
                                r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_SHIFTED_ORDER'])**2
                                var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                                estimations.append({
                                        "n": n,
                                        "sim": file.split('_')[1].split('.')[0],
                                        "noise_level": noise,
                                        "ate_classic RMST": ate_rmst_manual,
                                        "var_classic RMST": var_ate_rmst_manual,
                                        "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                                        "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                                        "r2_shifted_order RMST": r_2_shifted_order,
                                        "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                                        "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                                        })
                                df_sim.append(df_tmp)

        df_sim = pd.concat(df_sim, ignore_index=True)
        estimations = pd.DataFrame(estimations)
        df_sim.to_csv(f"sim_outputs_1000/df_sim_fc{fc}_ve{ve}_counter.csv", index=False)
        estimations.to_csv(f"sim_outputs_1000/estimations_fc{fc}_ve{ve}_counter.csv", index=False)
